In [1]:
import os
import glob
import pympi

In [ ]:
def validate_dataset_pairing(elan_dir, video_dir):
    """
    Performs a bi-directional check between ELAN files and Video files.
    Ensures that tiers and video suffixes match perfectly.
    """
    print(f"🔍 Starting Dataset Validation...")
    print(f"   ELAN Dir:  {elan_dir}")
    print(f"   Video Dir: {video_dir}\n")

    # 1. Gather all files
    eaf_files = glob.glob(os.path.join(elan_dir, '**/*.eaf'), recursive=True)
    video_files = glob.glob(os.path.join(video_dir, '**/*.mp4'), recursive=True)
    
    # Create clean dictionaries for fast lookup
    # Key: Base filename (e.g., "Session_01") -> Value: Full path
    video_dict = {}
    for vf in video_files:
        basename = os.path.basename(vf)
        # Extract the core name by stripping the known suffixes and extension
        core_name = basename.replace('_1a1.mp4', '').replace('_1b1.mp4', '')
        
        if core_name not in video_dict:
            video_dict[core_name] = []
        video_dict[core_name].append(basename)

    # Trackers for reporting
    missing_videos = []
    missing_elan = []
    missing_tiers = []

    # =========================================================================
    # PASS 1: Check ELAN -> Video
    # =========================================================================
    valid_eaf_core_names = {} # Store for Pass 2
    
    for eaf_path in eaf_files:
        basename = os.path.basename(eaf_path)
        core_name = basename.replace('.eaf', '')
        
        try:
            eaf = pympi.Elan.Eaf(eaf_path)
            tiers = list(eaf.get_tier_names())
            valid_eaf_core_names[core_name] = tiers
            
            # Check for Sign_r_A
            if "Sign_r_A" in tiers:
                expected_video = f"{core_name}_1a1.mp4"
                if core_name not in video_dict or expected_video not in video_dict[core_name]:
                    missing_videos.append(f"[Missing Video] {expected_video} (Needed by {basename})")
                    
            # Check for Sign_r_B
            if "Sign_r_B" in tiers:
                expected_video = f"{core_name}_1b1.mp4"
                if core_name not in video_dict or expected_video not in video_dict[core_name]:
                    missing_videos.append(f"[Missing Video] {expected_video} (Needed by {basename})")
                    
        except Exception as e:
            print(f"⚠️ Could not parse {basename}: {e}")

    # =========================================================================
    # PASS 2: Check Video -> ELAN
    # =========================================================================
    for core_name, v_files in video_dict.items():
        expected_eaf = f"{core_name}.eaf"
        
        # 1. Does the ELAN file exist at all?
        if core_name not in valid_eaf_core_names:
            for vf in v_files:
                missing_elan.append(f"[Missing ELAN] {expected_eaf} (Needed by {vf})")
            continue
            
        # 2. If it exists, does it have the correct tier for the video?
        tiers_in_eaf = valid_eaf_core_names[core_name]
        
        for vf in v_files:
            if vf.endswith("_1a1.mp4") and "Sign_r_A" not in tiers_in_eaf:
                missing_tiers.append(f"[Missing Tier] {expected_eaf} is missing 'Sign_r_A' (Needed by {vf})")
            elif vf.endswith("_1b1.mp4") and "Sign_r_B" not in tiers_in_eaf:
                missing_tiers.append(f"[Missing Tier] {expected_eaf} is missing 'Sign_r_B' (Needed by {vf})")

    # =========================================================================
    # PRINT REPORT
    # =========================================================================
    print("="*50)
    print("📊 VALIDATION REPORT")
    print("="*50)
    
    if not missing_videos and not missing_elan and not missing_tiers:
        print("✅ ALL CLEAR! Every ELAN file and Video file is perfectly paired.")
    else:
        if missing_videos:
            print(f"\n❌ MISSING VIDEOS ({len(missing_videos)}):")
            for msg in missing_videos: print(f"  - {msg}")
            
        if missing_elan:
            print(f"\n❌ MISSING ELAN FILES ({len(missing_elan)}):")
            for msg in missing_elan: print(f"  - {msg}")
            
        if missing_tiers:
            print(f"\n❌ MISSING TIERS IN ELAN ({len(missing_tiers)}):")
            for msg in missing_tiers: print(f"  - {msg}")
            
    print("="*50)

if __name__ == "__main__":
    # --- CONFIGURATION ---
    # Update these paths to point to your actual folders
    ELAN_DIRECTORY = "./raw_data/annotations" 
    VIDEO_DIRECTORY = "./raw_data/videos" 
    
    validate_dataset_pairing(ELAN_DIRECTORY, VIDEO_DIRECTORY)

In [3]:
def count_dataset_stats(elan_dir, video_dir, target_tiers):
    """
    Scans the directories to count total videos and extracts the exact 
    number of annotations from the target tiers inside the ELAN files.
    """
    print("📊 Scanning dataset directories...\n")

    # ==========================================
    # 1. Count Total Videos
    # ==========================================
    video_files = glob.glob(os.path.join(video_dir, '**/*.mp4'), recursive=True)
    total_videos = len(video_files)
    
    # ==========================================
    # 2. Parse ELAN Files & Count Annotations
    # ==========================================
    eaf_files = glob.glob(os.path.join(elan_dir, '**/*.eaf'), recursive=True)
    total_eafs = len(eaf_files)
    
    # Initialize counters for our target tiers
    tier_counts = {tier: 0 for tier in target_tiers}
    files_with_errors = 0
    
    for eaf_path in eaf_files:
        try:
            # Load the ELAN file using pympi
            eaf = pympi.Elan.Eaf(eaf_path)
            available_tiers = eaf.get_tier_names()
            
            # Count the individual annotations (gloss boundaries) in each tier
            for tier in target_tiers:
                if tier in available_tiers:
                    # get_annotation_data_for_tier returns a list of tuples for each tagged boundary
                    annotations = eaf.get_annotation_data_for_tier(tier)
                    tier_counts[tier] += 1
                    
        except Exception as e:
            print(f"⚠️ Error parsing {os.path.basename(eaf_path)}: {e}")
            files_with_errors += 1

    # ==========================================
    # 3. Print Final Report
    # ==========================================
    print("="*50)
    print("📋 DATASET VOLUME SUMMARY")
    print("="*50)
    print(f"🎬 Total Video Files (.mp4):  {total_videos:,}")
    print(f"📝 Total ELAN Files (.eaf):   {total_eafs:,}")
    
    if files_with_errors > 0:
        print(f"⚠️ ELAN Files with errors:    {files_with_errors}")
        
    print("-" * 50)
    print("📌 ANNOTATIONS BY TARGET TIER:")
    
    total_annotations = 0
    for tier, count in tier_counts.items():
        print(f"  -> '{tier}': {count:,} boundaries")
        total_annotations += count
        
    print("-" * 50)
    print(f"🎯 TOTAL COMBINED ANNOTATIONS: {total_annotations:,}")
    print("="*50)

if __name__ == "__main__":
    # --- CONFIGURATION ---
    # Update these paths to point to your actual folders
    ELAN_DIRECTORY = "./raw_data/annotations" 
    VIDEO_DIRECTORY = "./raw_data/videos" 
    
    # The specific tiers we want to extract counts from
    TARGET_TIERS = ["Sign_r_A", "Sign_r_B"] 
    
    count_dataset_stats(ELAN_DIRECTORY, VIDEO_DIRECTORY, TARGET_TIERS)

📊 Scanning dataset directories...

📋 DATASET VOLUME SUMMARY
🎬 Total Video Files (.mp4):  622
📝 Total ELAN Files (.eaf):   316
--------------------------------------------------
📌 ANNOTATIONS BY TARGET TIER:
  -> 'Sign_r_A': 281 boundaries
  -> 'Sign_r_B': 293 boundaries
--------------------------------------------------
🎯 TOTAL COMBINED ANNOTATIONS: 574
